# Data Cleaning


## Load Data

In [94]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
server = r"SIX\SQLEXPRESS"
database = "OlistDB"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

# Load các bảng chính (điều chỉnh path theo thư mục của bạn)
dataset_dir = r'C:\Users\nghah\OneDrive\Documents\dataset'

# Đọc các file CSV bằng cách nối thư mục gốc với tên file
orders = pd.read_csv(os.path.join(dataset_dir, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(dataset_dir, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(dataset_dir, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(dataset_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(dataset_dir, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(dataset_dir, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(dataset_dir, 'olist_sellers_dataset.csv'))
category = pd.read_csv(os.path.join(dataset_dir, 'product_category_name_translation.csv'))

# Convert timestamp columns sang datetime ngay từ đầu
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [1]:
%run "eda.ipynb"


orders: 99,441 rows x 8 columns
{dtype('<M8[ns]'): 5, dtype('O'): 3}
----------------------------------------
customers: 99,441 rows x 5 columns
{dtype('O'): 4, dtype('int64'): 1}
----------------------------------------
order_items: 112,650 rows x 7 columns
{dtype('O'): 4, dtype('float64'): 2, dtype('int64'): 1}
----------------------------------------
payments: 103,886 rows x 5 columns
{dtype('O'): 2, dtype('int64'): 2, dtype('float64'): 1}
----------------------------------------
reviews: 99,224 rows x 7 columns
{dtype('O'): 6, dtype('int64'): 1}
----------------------------------------
products: 32,951 rows x 9 columns
{dtype('float64'): 7, dtype('O'): 2}
----------------------------------------
sellers: 3,095 rows x 4 columns
{dtype('O'): 3, dtype('int64'): 1}
----------------------------------------
               Variable    Mean  Median  Std Dev   Min       Max  Skewness
0           price (BRL)  120.65   74.99   183.63  0.85   6735.00      7.92
1   freight_value (BRL)   19.99  

## CLEAN DATA

### Điền median cho kích thước sản phẩm

In [11]:
# Kiểm chứng TRƯỚC khi điền
print("NULL trước khi xử lý:")
print(products[['product_weight_g','product_length_cm',
                 'product_height_cm','product_width_cm']].isnull().sum())

for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)
    print(f"{col}: điền {products[col].isnull().sum()} NULL còn lại bằng median = {median_val}")

# Kiểm chứng SAU khi điền
print("\nNULL sau khi xử lý:")
print(products[['product_weight_g','product_length_cm',
                 'product_height_cm','product_width_cm']].isnull().sum())

NULL trước khi xử lý:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64
product_weight_g: điền 0 NULL còn lại bằng median = 700.0
product_length_cm: điền 0 NULL còn lại bằng median = 25.0
product_height_cm: điền 0 NULL còn lại bằng median = 13.0
product_width_cm: điền 0 NULL còn lại bằng median = 20.0

NULL sau khi xử lý:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


## Cap Price

In [12]:
# BEFORE: thống kê mô tả trước khi xử lý
print("--- TRƯỚC KHI XỬ LÝ OUTLIER ---")
print(order_items['price'].describe())

Q1 = order_items['price'].quantile(0.25)
Q3 = order_items['price'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f"\nQ1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, upper_bound={upper_bound:.2f}")

n_outliers = (order_items['price'] > upper_bound).sum()
n_extreme = (order_items['price'] > 5000).sum()
print(f"Số dòng vượt ngưỡng IQR: {n_outliers}")
print(f"Số dòng cực đoan (>5000 BRL): {n_extreme}")

# XỬ LÝ
order_items['price_capped'] = order_items['price'].clip(upper=upper_bound)
order_items = order_items[order_items['price'] <= 5000]

# AFTER
print("\n--- SAU KHI XỬ LÝ OUTLIER ---")
print(order_items['price_capped'].describe())

--- TRƯỚC KHI XỬ LÝ OUTLIER ---
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Q1=39.90, Q3=134.90, IQR=95.00, upper_bound=277.40
Số dòng vượt ngưỡng IQR: 8427
Số dòng cực đoan (>5000 BRL): 3

--- SAU KHI XỬ LÝ OUTLIER ---
count    112647.000000
mean         98.439457
std          75.918638
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max         277.400000
Name: price_capped, dtype: float64
